## Bu projede 3 ayri cesit misir cesidini farkli acilardan cekilmis 350 er adet resimle modele tanitip egittikten sonra ,streamlit uygulamali bir tahmin modeli gelistirecegiz. 

In [3]:
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D


In [4]:
ls

 Volume in drive C is OS
 Volume Serial Number is D881-D5E9

 Directory of C:\Users\osman\Documents\Yapay-Zeka-Kampi\ODEVLER\3 - CNN - Convolutional Neural Networks 8 DIFFERENT PROJECTS\2.proje

02/04/2026  03:35 PM    <DIR>          .
02/04/2026  03:28 PM    <DIR>          ..
02/04/2026  03:31 PM    <DIR>          .ipynb_checkpoints
02/04/2026  03:35 PM             4,504 Classification of Corn Varieties Using Deep Learning .ipynb
02/04/2026  03:26 PM    <DIR>          Corn_3_Classes_Image_Dataset
02/04/2026  02:45 PM             6,591 Explanation_Citation_Request.txt
               2 File(s)         11,095 bytes
               4 Dir(s)  182,966,525,952 bytes free


In [5]:
os.listdir("Classes_Corn3_Dataset")

['Zea_mays_Chulpi_Cancha', 'Zea_mays_Indurata', 'Zea_mays_Rugosa']

##### Klasorleri duzenle

In [15]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

##### Train_Validation_Sets

In [16]:
train_gen = datagen.flow_from_directory(
    "Classes_Corn3_Dataset",
    target_size=(128,128),
    batch_size=32,
    class_mode="sparse",
    subset="training",
    shuffle=True # egitim sirasinda model ezberlemez, surekli ardisk ornek gelmez.
)

val_gen = datagen.flow_from_directory(
    directory="Classes_Corn3_Dataset",
    target_size=(128, 128),
    batch_size=32,
    class_mode="sparse",
    subset="validation",
    shuffle=False # Her epoch’ta aynı örnekler, aynı sırayla gelir  Validation accuracy / loss kararlı olur  / Epoch’lar arası karşılaştırma adil olur
)

Found 840 images belonging to 3 classes.
Found 210 images belonging to 3 classes.


##### Model

In [17]:
model = Sequential([
    Input(shape=(128,128,3)),

    Conv2D(32, 3, activation='relu', padding='same'),
    MaxPooling2D(),

    Conv2D(64, 3, activation='relu', padding='same'),
    MaxPooling2D(),

    Conv2D(128, 3, activation='relu', padding='same'),
    GlobalAveragePooling2D(),

    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])


In [18]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [19]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [20]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,                 # 🔴 üst sınır
    callbacks=[early_stop]
)

Epoch 1/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 7s 191ms/step - accuracy: 0.4226 - loss: 0.9866 - val_accuracy: 0.6048 - val_loss: 0.8563
Epoch 2/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 5s 192ms/step - accuracy: 0.6643 - loss: 0.6577 - val_accuracy: 0.8143 - val_loss: 0.4444
Epoch 3/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 5s 199ms/step - accuracy: 0.7952 - loss: 0.4028 - val_accuracy: 0.8190 - val_loss: 0.3513
Epoch 4/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 7s 262ms/step - accuracy: 0.8524 - loss: 0.3095 - val_accuracy: 0.9000 - val_loss: 0.2572
Epoch 5/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 7s 267ms/step - accuracy: 0.8893 - loss: 0.2629 - val_accuracy: 0.8238 - val_loss: 0.3297
Epoch 6/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 6s 232ms/step - accuracy: 0.8810 - loss: 0.2711 - val_accuracy: 0.9429 - val_loss: 0.2077
Epoch 7/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 7s 244ms/step - accuracy: 0.9107 - loss: 0.2102 - val_accuracy: 0.8714 - val_loss: 0.2385
Epoch 8/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 6s 225ms/step - accuracy: 0.9488 - loss: 0.1692 - val_accuracy: 0.

In [21]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)                    │ (None, 128, 128, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 64, 64, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_9 (Conv2D)                    │ (None, 64, 64, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 32, 32, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_10 (Conv2D)                   │ (None, 32, 32, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 128)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 3)                   │             195 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 305,099 (1.16 MB)

 Trainable params: 101,699 (397.26 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 203,400 (794.54 KB)

In [22]:
model.save('corn_varieties_cnn.keras')

### Transfer Learning

In [23]:
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)


In [25]:
train_gen = datagen.flow_from_directory(
    "Classes_Corn3_Dataset",
    target_size=(170,170),
    batch_size=32,
    class_mode="sparse",
    subset="training",
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    "Classes_Corn3_Dataset",
    target_size=(170,170),
    batch_size=32,
    class_mode="sparse",
    subset="validation",
    shuffle=False
)


Found 840 images belonging to 3 classes.
Found 210 images belonging to 3 classes.


##### Base model

In [26]:
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(170, 170, 3)
)


##### Freeze

In [27]:
for layer in base_model.layers:
    layer.trainable = False

##### TL Modelling

In [28]:
model = Sequential([
    Input(shape=(170,170,3)),
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(3, activation="softmax")
])

In [29]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [30]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)


In [31]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,              # üst sınır
    callbacks=[early_stop]
)

Epoch 1/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 75s 3s/step - accuracy: 0.4190 - loss: 4.0533 - val_accuracy: 0.8571 - val_loss: 0.3034
Epoch 2/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - accuracy: 0.7393 - loss: 1.2883 - val_accuracy: 0.9857 - val_loss: 0.0470
Epoch 3/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 80s 3s/step - accuracy: 0.8310 - loss: 0.7188 - val_accuracy: 0.9952 - val_loss: 0.0275
Epoch 4/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 80s 3s/step - accuracy: 0.9048 - loss: 0.3766 - val_accuracy: 0.9952 - val_loss: 0.0264
Epoch 5/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 85s 3s/step - accuracy: 0.9262 - loss: 0.3239 - val_accuracy: 0.9952 - val_loss: 0.0202
Epoch 6/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 94s 4s/step - accuracy: 0.9560 - loss: 0.1917 - val_accuracy: 0.9952 - val_loss: 0.0202
Epoch 7/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 79s 3s/step - accuracy: 0.9560 - loss: 0.1657 - val_accuracy: 0.9952 - val_loss: 0.0206


In [32]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)                   │ (None, 5, 5, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_2           │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 3)                   │             771 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 15,110,987 (57.64 MB)

 Trainable params: 132,099 (516.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

 Optimizer params: 264,200 (1.01 MB)

In [33]:
model.save("corn_vgg16_transfer.keras")